# Public light viewer

This derives a **display-only approximation** of the M07 gross lung and airway mesh for a faster public web entry point. It does not change the deposited geometry, source model, scientific coordinates, independent E02 ROI, or full-quality viewer. A voxel-clustered surface can change fine contours and 2D plane intersections. Lite allows 3D clipping but suppresses the 2D section. Download and open the full HTML locally for detailed work.


In [1]:
from pathlib import Path
import base64, gzip, json, re
import numpy as np

ROOT = Path.cwd()
SOURCE = ROOT / 'viewer/mouse_lung_v3.2.html'
OUTPUT = ROOT / 'viewer/mouse_lung_v3.2_lite.html'
GRID_MM = 0.08
source_html = SOURCE.read_text()
match = re.search(r'(<script id="meshPayload" type="application/json">)(.*?)(</script>)', source_html, re.S)
assert match, 'M07 mesh payload not found'
meshes = json.loads(match.group(2))
assert len(meshes) == 6
print(f'Source: {SOURCE.name}; {len(meshes)} meshes; {SOURCE.stat().st_size/2**20:.1f} MiB')


Source: mouse_lung_v3.2.html; 6 meshes; 42.8 MiB


## Cluster and reindex each surface

Vertices are grouped in 0.08 mm cubic cells in the unchanged M07 coordinate frame. Each cluster is represented by its mean position. Degenerate and duplicate triangles are removed; orientation is retained from the source face. This is a visual level of detail, **not an anatomical remeshing or segmentation**.


In [2]:
def decode_array(text, dtype, columns):
    return np.frombuffer(gzip.decompress(base64.b64decode(text)), dtype=dtype).reshape(-1, columns)

def encode_array(a):
    return base64.b64encode(gzip.compress(a.tobytes(order='C'), compresslevel=6, mtime=0)).decode('ascii')

def clustered_mesh(d, grid_mm):
    q = decode_array(d['q_b64'], '<u2', 3)
    faces = decode_array(d['i_b64'], '<u4', 3)
    lo, hi = np.asarray(d['min'], dtype=np.float64), np.asarray(d['max'], dtype=np.float64)
    pos = lo + q.astype(np.float64) * ((hi-lo)/65535.0)
    cells = np.floor(pos/grid_mm).astype(np.int32)
    _, inverse = np.unique(cells, axis=0, return_inverse=True)
    count = np.bincount(inverse)
    clustered = np.column_stack([np.bincount(inverse, weights=pos[:,axis])/count for axis in range(3)])
    mapped = inverse[faces]
    valid = (mapped[:,0] != mapped[:,1]) & (mapped[:,0] != mapped[:,2]) & (mapped[:,1] != mapped[:,2])
    mapped = mapped[valid]
    _, first = np.unique(np.sort(mapped, axis=1), axis=0, return_index=True)
    mapped = mapped[np.sort(first)]
    used, remapped = np.unique(mapped, return_inverse=True)
    clustered = clustered[used]
    mapped = remapped.reshape(-1,3).astype('<u4')
    qlite = np.clip(np.rint((clustered-lo)/(hi-lo)*65535),0,65535).astype('<u2')
    out = {k:d[k] for k in ('name','color','min','max')}
    out.update(verts=len(qlite), faces=len(mapped), q_b64=encode_array(qlite), i_b64=encode_array(mapped))
    return out

light_meshes = [clustered_mesh(d, GRID_MM) for d in meshes]
for old,new in zip(meshes,light_meshes):
    print(f"{old['name']}: {old['verts']:,} -> {new['verts']:,} vertices; {old['faces']:,} -> {new['faces']:,} faces")


Left lobe: 375,587 -> 72,279 vertices; 751,306 -> 144,649 faces
Right accessory lobe: 226,549 -> 43,292 vertices; 453,442 -> 86,876 faces
Right caudal lobe: 338,500 -> 65,530 vertices; 677,032 -> 131,175 faces
Right cranial lobe: 266,251 -> 50,436 vertices; 533,222 -> 101,214 faces
Right middle lobe: 213,127 -> 40,842 vertices; 426,626 -> 82,024 faces
Trachea and segmented airway surface: 551,640 -> 97,936 vertices; 1,103,154 -> 197,332 faces


## Publish a clearly labeled interactive build

The viewer UI and interaction code remain the same. The independent measured E02 ROI is retained unmodified. We add a visible notice and change the document title to identify the approximate web build. Capture buttons are hidden and replaced by a Full HTML download prompt.


In [3]:
replacement = json.dumps(light_meshes, separators=(',',':'))
light_html = source_html[:match.start(2)] + replacement + source_html[match.end(2):]
light_html = light_html.replace('<title>Mouse Lung Explorer 3.2 | M07 model · NAV anatomy</title>', '<title>Mouse Lung Explorer 3.2 Lite | M07 display approximation</title>', 1)
light_html = light_html.replace('<h1>Mouse lung explorer · v3.2</h1>', '<h1>Mouse lung explorer · v3.2 Lite</h1>', 1)
notice = '<p class="liteNotice" style="font-size:11px;line-height:1.45;margin:8px 0;color:#a8e3f1;border-left:2px solid #7bd9f3;padding-left:9px">Fast public preview · 0.08 mm clustered display mesh. Rotate and clip the 3D surface here. For an accurate 2D section and all exports, download and open the Full HTML locally.</p>'
light_html = light_html.replace('<h1>Mouse lung explorer · v3.2 Lite</h1>', '<h1>Mouse lung explorer · v3.2 Lite</h1>'+notice, 1)
light_html = light_html.replace("'Mouse lung explorer · v3.2'", "'Mouse lung explorer · v3.2 Lite'")
light_html = light_html.replace('function scheduleSlice(){', 'function scheduleSlice(){return;', 1)
# Keep motion controls, but route every capture request to the full downloadable build.
light_css = """<style id="liteCaptureGateStyle">
#saveLungPng,#recordLungGif,#recordLungVideo,#gifTimer,#pngSetup,#gifSetup,#videoSetup,#saveRoiPng,#recordRoiVideo,#slicePanel,#clipInset{display:none!important}
#liteExportGate{position:relative;color:#e3eff6;font:11px system-ui,sans-serif}
#liteExportGate summary{list-style:none;cursor:pointer;white-space:nowrap;border:1px solid #355668;border-radius:10px;padding:8px 10px;background:#10212b}
#liteExportGate summary::-webkit-details-marker{display:none}
#liteExportGate[open] summary{border-color:#7bd9f3}
#liteExportGate .liteGateBody{position:absolute;right:0;top:calc(100% + 8px);z-index:80;width:min(280px,calc(100vw - 24px));padding:14px;border:1px solid #355668;border-radius:12px;background:#0c1821;box-shadow:0 12px 30px #0009;line-height:1.45}
#liteExportGate .liteGateBody p{color:#bdcbd5;margin:8px 0 12px}
#liteExportGate .liteGateBody a{display:inline-block;color:#091923;background:#a8e3f1;border-radius:8px;padding:8px 10px;text-decoration:none;font-weight:700}
@media(max-width:760px){#liteExportGate .liteGateBody{top:auto;bottom:calc(100% + 8px)}}
</style>"""
light_html = light_html.replace('</head>', light_css+'</head>', 1)
light_gate = """<details id="liteExportGate"><summary>Record / export</summary><div class="liteGateBody"><strong>Full viewer required</strong><p>To save a PNG, GIF, or video, download the full-quality HTML and open it in your browser.</p><a href="mouse_lung_v3.2.html" download="mouse-lung-explorer-v3.2.html">Download full viewer</a></div></details>"""
light_html = light_html.replace('  <div id="presentationToolbar" role="toolbar" aria-label="Presentation export and motion">', '  <div id="presentationToolbar" role="toolbar" aria-label="Motion and full-viewer download">'+light_gate, 1)
assert 'id="liteExportGate"' in light_html

assert 'v3.2 Lite' in light_html and len(light_html) < len(source_html)
OUTPUT.write_text(light_html)
report = {'source':str(SOURCE.relative_to(ROOT)),'output':str(OUTPUT.relative_to(ROOT)),'grid_mm':GRID_MM,'source_bytes':SOURCE.stat().st_size,'light_bytes':OUTPUT.stat().st_size,'source_vertices':sum(d['verts'] for d in meshes),'light_vertices':sum(d['verts'] for d in light_meshes),'source_faces':sum(d['faces'] for d in meshes),'light_faces':sum(d['faces'] for d in light_meshes),'purpose':'display-only approximation; 3D clip allowed, 2D section and all exports disabled'}
(ROOT/'qa/public_light_viewer_report.json').write_text(json.dumps(report,indent=2)+'\n')
print(f"Light HTML: {OUTPUT.stat().st_size/2**20:.1f} MiB ({100*OUTPUT.stat().st_size/SOURCE.stat().st_size:.1f}% of full)")


Light HTML: 18.1 MiB (42.3% of full)


## Validation

Check the generated HTML in a browser: visible lite label; mouse-lung model rendering; rotation and plane movement; functional 3D Clip with 2D section hidden; no visible capture controls; Record / export opens a full-viewer download message; no console errors. Compare screenshots at the same plane with the full viewer. This is a display-quality check only.
